<a href="https://colab.research.google.com/github/soroushfazeli95/Crafter-AI-Lab/blob/main/Notebooks/01_crafter_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crafter setup

Run all cells in a CPU Colab runtime to install Crafter, see an initial observation, and watch a random policy act live.

## Install Crafter

In [ ]:
%pip install -q crafter==1.8.3

## Start an episode

`reset()` returns the RGB image available to the agent.

In [ ]:
import crafter
import matplotlib.pyplot as plt
import numpy as np

env = crafter.Env()
observation = env.reset()

plt.imshow(observation, interpolation="nearest")
plt.title("Initial observation")
plt.axis("off")
plt.show()

## Watch a policy live

`watch_policy(env, policy, max_steps=100, fps=10)` starts a fresh episode, updates one image inline, and returns rollout stats. The policy receives the current RGB observation and returns one integer action. To watch a trained agent later, pass a callable that runs your model on that observation and returns its chosen action.

Use `fps` to set the playback speed and `max_steps` to limit the rollout. Playback can be slower if the policy or environment takes longer than the frame interval.

In [ ]:
from time import monotonic, sleep

from IPython.display import clear_output, display
from PIL import Image


def watch_policy(env, policy, *, max_steps=100, fps=10):
    """Display a live Crafter rollout and return its stats."""
    if max_steps < 1:
        raise ValueError("max_steps must be at least 1")
    if fps <= 0:
        raise ValueError("fps must be positive")

    def frame(observation):
        image = Image.fromarray(observation)
        return image.resize((image.width * 5, image.height * 5), Image.Resampling.NEAREST)

    observation = env.reset()
    display(frame(observation))
    total_reward = 0.0

    for steps in range(1, max_steps + 1):
        started = monotonic()
        action = int(policy(observation))
        observation, reward, done, info = env.step(action)
        total_reward += reward
        image = frame(observation)
        sleep(max(0.0, 1 / fps - (monotonic() - started)))
        clear_output(wait=True)
        display(image)
        if done:
            break

    return {
        "steps": steps,
        "total_reward": total_reward,
        "unlocked_achievements": {
            name: count for name, count in info["achievements"].items() if count > 0
        },
    }

## Watch random actions

The random policy samples uniformly from Crafter's action space. The rollout takes up to 100 steps and stops early if the episode ends.

In [ ]:
rng = np.random.default_rng()


def random_policy(observation):
    return int(rng.integers(env.action_space.n))


stats = watch_policy(env, random_policy)
print(f"Steps: {stats['steps']}")
print(f"Total reward: {stats['total_reward']:.2f}")
print(f"Unlocked achievements: {stats['unlocked_achievements'] or 'None'}")